# 🏦 Bank Customer Churn — Visual EDA & Business Insights

**Dataset :** [Bank Customer Churn Prediction — Kaggle](https://www.kaggle.com/datasets/shantanudhakadd/bank-customer-churn-prediction)  
**Stack    :** Python · Pandas · Matplotlib · Seaborn  
**Goal     :** Visual storytelling to surface actionable business insights — no ML models.

---
### 📖 Story Arc
| Act | Question | Chart |
|-----|----------|-------|
| 1 | What variables correlate with churn? | Correlation Heatmap |
| 2 | *Who* is churning? | Demographics — Geography & Gender |
| 3 | *How much* money is walking out the door? | Financial Distributions |

> ⚠️ **Before running:** Download `Churn_Modelling.csv` from Kaggle and place it in the same folder as this notebook (or upload it to your Colab session).

---
## CELL 1 — Imports & Global Style Configuration
Set once here — applies to every chart. Keeps the notebook visually cohesive.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Colour palette ──────────────────────────────────────────────────────
PALETTE_CHURN = {0: "#2196F3", 1: "#F44336"}   # Blue = Stayed, Red = Churned
PALETTE_GEO   = ["#1565C0", "#E53935", "#2E7D32"]
FONT_TITLE    = {"fontsize": 16, "fontweight": "bold", "color": "#1A1A2E"}
FONT_LABEL    = {"fontsize": 12, "color": "#333333"}
SPINE_COLOR   = "#CCCCCC"
BG_COLOR      = "#FAFAFA"

# ── Seaborn base theme ──────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font="DejaVu Sans")
plt.rcParams.update({
    "figure.facecolor"  : BG_COLOR,
    "axes.facecolor"    : BG_COLOR,
    "grid.color"        : "#E8E8E8",
    "grid.linewidth"    : 0.8,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

# ── Reusable helper: spine & label polish ───────────────────────────────
def style_axes(ax, xlabel="", ylabel="", title=""):
    """Apply consistent professional polish to any Axes object."""
    ax.set_title(title, **FONT_TITLE, pad=14)
    ax.set_xlabel(xlabel, **FONT_LABEL, labelpad=10)
    ax.set_ylabel(ylabel, **FONT_LABEL, labelpad=10)
    ax.tick_params(axis="both", labelsize=10, colors="#444444")
    ax.spines["left"].set_color(SPINE_COLOR)
    ax.spines["bottom"].set_color(SPINE_COLOR)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return ax

print("✅  Libraries loaded & global styles set.")

---
## CELL 2 — Load & Inspect the Dataset

**WHY:** Always validate shape, dtypes, and churn rate before plotting.  
A quick `.describe()` catches data quality issues early.

In [ ]:
df = pd.read_csv("Churn_Modelling.csv")

print(f"Shape      : {df.shape}")
print(f"Churn Rate : {df['Exited'].mean():.1%}  ({df['Exited'].sum()} customers churned)")
print("\nColumn dtypes:")
print(df.dtypes)
print("\n── First 5 rows ──")
df.head()

---
## CELL 3 — Descriptive Statistics

**WHY:** Check skewness of Balance and Salary before choosing chart types.  
Skewed distributions need violin/box plots — not histograms — to avoid misleading visuals.

In [ ]:
ANALYSIS_COLS = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "IsActiveMember", "EstimatedSalary", "Exited"
]

desc = df[ANALYSIS_COLS].describe().T
desc["skewness"] = df[ANALYSIS_COLS].skew()
desc["kurtosis"] = df[ANALYSIS_COLS].kurtosis()
desc.round(3)

---
## ACT 1 — CELL 4 : Correlation Matrix Heatmap

**Question:** *What financial and behavioural signals correlate with churn?*

**WHY:** Pearson correlation rules out irrelevant features and spotlights real drivers fast.  
Lower-triangle mask removes redundant mirror values — a small but important readability choice.

In [ ]:
NUMERIC_COLS = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember",
    "EstimatedSalary", "Exited"
]

corr_matrix = df[NUMERIC_COLS].corr()

# Clean axis labels
label_map = {
    "CreditScore"    : "Credit Score",
    "Age"            : "Age",
    "Tenure"         : "Tenure (yrs)",
    "Balance"        : "Account Balance",
    "NumOfProducts"  : "# Products",
    "HasCrCard"      : "Has Credit Card",
    "IsActiveMember" : "Active Member",
    "EstimatedSalary": "Est. Salary",
    "Exited"         : "Churn",
}
corr_matrix = corr_matrix.rename(index=label_map, columns=label_map)

# Lower-triangle mask — hide redundant upper half
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor(BG_COLOR)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor="#E0E0E0",
    square=True,
    cbar_kws={"shrink": 0.75, "label": "Pearson r"},
    annot_kws={"size": 9},
    ax=ax,
)

ax.set_title(
    "Correlation Matrix — Financial & Behavioural Variables vs. Churn",
    **FONT_TITLE, pad=18
)
ax.tick_params(axis="x", rotation=35, labelsize=9)
ax.tick_params(axis="y", rotation=0,  labelsize=9)

plt.tight_layout()
plt.savefig("chart1_correlation_matrix.png", dpi=180, bbox_inches="tight")
plt.show()
print("✅  Chart 1 saved → chart1_correlation_matrix.png")

### 💡 Business Insight — Correlation Matrix

> **Age** has the strongest positive correlation with churn (~0.29), meaning older customers are the highest flight-risk segment — a clear signal for targeted retention outreach.  
> **Active Member** shows the strongest *negative* correlation (~-0.16), confirming engagement is a powerful churn shield; the bank should prioritise activation campaigns for dormant accounts before they leave.

---
## ACT 2 — CELL 5 : Demographic Risk Factors

**Question:** *Who exactly is churning — by Geography and Gender?*

**WHY:** The aggregate churn rate (20%) hides massive variation across segments.  
The dashed benchmark line gives every bar instant context without needing a separate table.

In [ ]:
# Pre-compute churn rates per group
geo_churn = (
    df.groupby("Geography")["Exited"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "ChurnRate", "sum": "Churned", "count": "Total"})
    .reset_index()
    .sort_values("ChurnRate", ascending=False)
)
geo_churn["ChurnPct"] = (geo_churn["ChurnRate"] * 100).round(1)

gender_churn = (
    df.groupby("Gender")["Exited"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "ChurnRate", "sum": "Churned", "count": "Total"})
    .reset_index()
)
gender_churn["ChurnPct"] = (gender_churn["ChurnRate"] * 100).round(1)

overall_churn = df["Exited"].mean() * 100

print("Geography breakdown:")
print(geo_churn[["Geography", "Churned", "Total", "ChurnPct"]].to_string(index=False))
print("\nGender breakdown:")
print(gender_churn[["Gender", "Churned", "Total", "ChurnPct"]].to_string(index=False))
print(f"\nOverall churn rate: {overall_churn:.1f}%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle("Demographic Risk Factors: Who Is Churning?", **FONT_TITLE, y=1.02)

# ── Panel A: Geography ───────────────────────────────────────────────────
bars = ax1.bar(
    geo_churn["Geography"],
    geo_churn["ChurnPct"],
    color=PALETTE_GEO,
    width=0.55,
    edgecolor="white",
    linewidth=1.2,
)

for bar, pct in zip(bars, geo_churn["ChurnPct"]):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.4,
        f"{pct}%",
        ha="center", va="bottom",
        fontsize=12, fontweight="bold", color="#1A1A2E"
    )

ax1.axhline(overall_churn, color="#FF6F00", linewidth=1.5,
            linestyle="--", label=f"Overall avg: {overall_churn:.1f}%")
ax1.legend(fontsize=9, frameon=False)
ax1.set_ylim(0, geo_churn["ChurnPct"].max() + 6)
style_axes(ax1, xlabel="Geography", ylabel="Churn Rate (%)",
           title="Churn Rate by Geography")

# ── Panel B: Gender ──────────────────────────────────────────────────────
bars2 = ax2.bar(
    gender_churn["Gender"],
    gender_churn["ChurnPct"],
    color=[PALETTE_CHURN[0], PALETTE_CHURN[1]],
    width=0.4,
    edgecolor="white",
    linewidth=1.2,
)

for bar, pct in zip(bars2, gender_churn["ChurnPct"]):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.4,
        f"{pct}%",
        ha="center", va="bottom",
        fontsize=12, fontweight="bold", color="#1A1A2E"
    )

ax2.axhline(overall_churn, color="#FF6F00", linewidth=1.5,
            linestyle="--", label=f"Overall avg: {overall_churn:.1f}%")
ax2.legend(fontsize=9, frameon=False)
ax2.set_ylim(0, gender_churn["ChurnPct"].max() + 6)
style_axes(ax2, xlabel="Gender", ylabel="Churn Rate (%)",
           title="Churn Rate by Gender")

plt.tight_layout()
plt.savefig("chart2_demographic_risk.png", dpi=180, bbox_inches="tight")
plt.show()
print("✅  Chart 2 saved → chart2_demographic_risk.png")

### 💡 Business Insight — Demographics

> **Germany** shows a dramatically higher churn rate (~32%) compared to France and Spain (~16%), suggesting a market-specific issue — stronger local competition, product-market misfit, or a weaker customer service operation in that region.  
> **Female customers** churn at a higher rate than males (~25% vs ~16%), indicating the bank's current product offerings or engagement strategies may not adequately address women's financial needs.

---
## ACT 3 — CELL 6 : Financial Distributions

**Question:** *How much money is leaving — and who are these customers financially?*

**WHY:**  
- `violinplot` with `inner='quartile'` shows the full distribution shape **and** Q1/median/Q3 — more information than a boxplot alone.  
- `boxplot` on Credit Score directly tests the assumption that low creditworthiness drives churn.

In [ ]:
df["Churn_Label"] = df["Exited"].map({0: "Retained", 1: "Churned"})

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle("Financial Profile: Churned vs. Retained Customers", **FONT_TITLE, y=1.02)

# ── Panel A: Violin plot — Account Balance ───────────────────────────────
ax = axes[0]
sns.violinplot(
    data=df,
    x="Churn_Label",
    y="Balance",
    hue="Churn_Label",
    palette={"Retained": "#2196F3", "Churned": "#F44336"},
    order=["Retained", "Churned"],
    inner="quartile",
    linewidth=1.2,
    legend=False,
    ax=ax,
)

for i, label in enumerate(["Retained", "Churned"]):
    median_val = df[df["Churn_Label"] == label]["Balance"].median()
    ax.text(
        i, median_val + 5000,
        f"Median\n${median_val:,.0f}",
        ha="center", fontsize=9, fontweight="bold", color="white",
        bbox=dict(boxstyle="round,pad=0.3", fc="#333333", alpha=0.75)
    )

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
style_axes(ax, xlabel="Customer Status", ylabel="Account Balance (USD)",
           title="Account Balance Distribution")

# ── Panel B: Box plot — Credit Score ────────────────────────────────────
ax2 = axes[1]
sns.boxplot(
    data=df,
    x="Churn_Label",
    y="CreditScore",
    hue="Churn_Label",
    palette={"Retained": "#2196F3", "Churned": "#F44336"},
    order=["Retained", "Churned"],
    width=0.45,
    linewidth=1.3,
    flierprops=dict(marker="o", markerfacecolor="#AAAAAA", markersize=3, alpha=0.4),
    legend=False,
    ax=ax2,
)

for i, label in enumerate(["Retained", "Churned"]):
    med = df[df["Churn_Label"] == label]["CreditScore"].median()
    ax2.text(
        i, med + 10,
        f"Median: {int(med)}",
        ha="center", fontsize=9, fontweight="bold", color="white",
        bbox=dict(boxstyle="round,pad=0.3", fc="#333333", alpha=0.75)
    )

style_axes(ax2, xlabel="Customer Status", ylabel="Credit Score",
           title="Credit Score Distribution")

plt.tight_layout()
plt.savefig("chart3_financial_distributions.png", dpi=180, bbox_inches="tight")
plt.show()
print("✅  Chart 3 saved → chart3_financial_distributions.png")

### 💡 Business Insight — Financial Distributions

> Churned customers hold **significantly higher account balances** than retained ones — the bank is losing its wealthiest customers, representing a disproportionate loss of Assets Under Management (AUM), not just headcount.  
> Credit score distributions are **nearly identical** for both groups, debunking the assumption that low creditworthiness drives churn — the root cause is engagement and experience, not financial risk.

---
## 📊 Final Summary — Key Findings

| # | Finding | Business Implication |
|---|---------|----------------------|
| 1 | **Age** is the top churn predictor (r ≈ 0.29) | Prioritise retention outreach for customers 45+ |
| 2 | **Active membership** is the strongest protective factor | Launch re-engagement campaigns for dormant accounts |
| 3 | **Germany** churns at 2× the rate of France/Spain | Conduct market-specific diagnosis — product, pricing, or ops |
| 4 | **Female customers** churn ~9pp more than males | Review whether product suite meets women's financial priorities |
| 5 | **Churned customers hold higher balances** | This is an AUM crisis, not a credit risk problem — fix the experience |
| 6 | **Credit score is irrelevant** to churn | Drop it from retention models; it adds noise, not signal |

---
*All charts saved as 180 dpi PNGs — ready for LinkedIn screenshots.*